In [ ]:
# === Colab / local bootstrap (run first) ===
import os
import subprocess
import sys
from pathlib import Path

GIT_REPO = "https://github.com/IronYR/llm-project.git"
GIT_BRANCH = "feat/v1"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.chdir("/content")
    root = Path("/content/llm-project")
    if not (root / "config.yaml").is_file():
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_REPO, str(root)],
            check=True,
        )
    os.chdir(root)
else:
    p = Path.cwd().resolve()
    if p.name == "notebooks":
        os.chdir(p.parent)
    elif not (p / "config.yaml").is_file() and (p.parent / "config.yaml").is_file():
        os.chdir(p.parent)

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

if IN_COLAB:
    from src.colab_setup import colab_pip_install
    colab_pip_install(ROOT)

print("ROOT =", ROOT)


ROOT = /content/llm-project


In [2]:
from src.colab_setup import ensure_knowledge_base

ensure_knowledge_base(ROOT)

Upload the course dataset files:
  - NUST Bank-Product-Knowledge.xlsx
  - funds_transfer_app_features_faq.json



Saving funds_transfer_app_features_faq.json to funds_transfer_app_features_faq.json
Wrote /content/llm-project/data/raw/funds_transfer_app_features_faq.json


KeyboardInterrupt: 

# LoRA fine-tuning (PEFT + TRL + bitsandbytes)

**This repo uses:** causal LM **Qwen2.5-3B-Instruct**, **LoRA** via `peft`, **SFTTrainer** from `trl`, optional **4-bit** loads with `bitsandbytes` on CUDA — not Flan-T5 / Seq2Seq.

**Before running:** `python ingest.py` and install deps (`pip install -r requirements.txt`). **GPU with CUDA** strongly recommended for 4-bit training.

## Load config and training data

In [2]:
from src.finetune_lib import (
    attach_lora,
    build_dataset_rows,
    build_sft_dataset,
    build_sft_training_args,
    load_base_model_for_training,
    load_config,
    load_tokenizer,
)

cfg = load_config(ROOT / "config.yaml")
ft = cfg.get("finetune", {})
data_cfg = cfg.get("data", {})
model_id = ft.get("base_model_id", "Qwen/Qwen2.5-3B-Instruct")
out_dir = Path(ft.get("output_dir", "./models/lora_nust_bank"))
out_dir.mkdir(parents=True, exist_ok=True)

rows = build_dataset_rows(data_cfg["processed_dir"], data_cfg["knowledge_base_file"])
print(f"Training examples: {len(rows)}")
assert rows, "No rows — run: python ingest.py"

tokenizer = load_tokenizer(model_id)
train_ds = build_sft_dataset(tokenizer, rows)
train_ds

Training examples: 319


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset({
    features: ['text'],
    num_rows: 319
})

## Load base model and attach LoRA

On **CUDA**, `use_4bit: true` in `config.yaml` loads the model in 4-bit. Otherwise the model loads in fp16 on CPU (or MPS if `use_mps: true`).

In [4]:
model = load_base_model_for_training(model_id, ft)
model = attach_lora(model, ft)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## Train with SFTTrainer

In [5]:
from trl import SFTTrainer

sft_cfg = build_sft_training_args(ft, out_dir)

try:
    trainer = SFTTrainer(
        model=model,
        args=sft_cfg,
        train_dataset=train_ds,
        processing_class=tokenizer,
    )
except TypeError:
    trainer = SFTTrainer(
        model=model,
        args=sft_cfg,
        train_dataset=train_ds,
        tokenizer=tokenizer,
    )

# First step can take a long time on CPU
trainer.train()

Adding EOS to train dataset:   0%|          | 0/319 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/319 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,3.983874
10,1.763387
20,0.607983


Step,Training Loss
1,3.983874
10,1.763387
20,0.607983
30,0.487537
40,0.423715
50,0.448829
60,0.381229
70,0.440984
80,0.371068


TrainOutput(global_step=80, training_loss=0.643347617983818, metrics={'train_runtime': 2623.0091, 'train_samples_per_second': 0.243, 'train_steps_per_second': 0.03, 'total_flos': 3784066251620352.0, 'train_loss': 0.643347617983818})

## Save adapter and tokenizer

In [6]:
trainer.save_model(str(out_dir))
tokenizer.save_pretrained(str(out_dir))
print("Saved to", out_dir.resolve())

Saved to /content/llm-project/models/lora_nust_bank
